# ByteTrack en Jupyter Notebook

Plantilla inicial para ejecutar seguimiento multiobjeto con **YOLO + ByteTrack** sobre un video.

> ByteTrack asocia detecciones entre fotogramas; YOLO proporciona las detecciones. Ejecuta las celdas en orden. Si ya tienes una implementación propia de ByteTrack, este notebook no la sustituye: sirve como punto de partida funcional.

## 1. Instalar dependencias
Ejecuta esta celda una vez en el entorno de Jupyter que usarás.

In [1]:
%pip install -U ultralytics opencv-python

Note: you may need to restart the kernel to use updated packages.


## 2. Configurar el video
Cambia `VIDEO_PATH` por la ruta real de tu video. En macOS puedes usar una ruta como `/Users/tu_usuario/Movies/video.mp4`.

In [2]:
from pathlib import Path
import cv2
from ultralytics import YOLO
from IPython.display import Video, display

VIDEO_PATH = Path("input1.mp4")  # <-- CAMBIAR
OUTPUT_PATH = Path("bytetrack_output.mp4")

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(
        f"No encuentro el video: {VIDEO_PATH}\n"
        "Edita VIDEO_PATH con la ruta real del archivo."
    )

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"Resolución: {width}x{height}")
print(f"FPS: {fps:.2f}")
print(f"Fotogramas: {frame_count}")
print(f"Duración aproximada: {frame_count / fps:.1f} s" if fps > 0 else "FPS no válido")

Resolución: 3840x2160
FPS: 59.94
Fotogramas: 2568
Duración aproximada: 42.8 s


## 3. Cargar YOLO
La primera ejecución descargará los pesos del modelo si no están disponibles localmente. `yolo11n.pt` es una variante pequeña para una prueba inicial.

In [3]:
model = YOLO("yolo11n.pt")
print("Modelo cargado correctamente.")

Modelo cargado correctamente.


## 4. Ejecutar ByteTrack y guardar el video
`conf=0.1` permite conservar detecciones de confianza relativamente baja, algo útil para ByteTrack. Si hay demasiadas detecciones falsas, prueba subirlo a `0.2` o `0.25`.

In [4]:
# Códec de salida MP4
writer = cv2.VideoWriter(
    str(OUTPUT_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps if fps > 0 else 30.0,
    (width, height)
)
if not writer.isOpened():
    raise RuntimeError("No se pudo crear el video de salida.")

frames_processed = 0
try:
    results_stream = model.track(
        source=str(VIDEO_PATH),
        tracker="bytetrack.yaml",
        persist=True,
        stream=True,
        conf=0.15,
        imgsz=960,
        verbose=False,
        save=False
    )

    for result in results_stream:
        annotated_frame = result.plot()  # BGR, con cajas y tracking IDs
        if annotated_frame.shape[1] != width or annotated_frame.shape[0] != height:
            annotated_frame = cv2.resize(annotated_frame, (width, height))
        writer.write(annotated_frame)
        frames_processed += 1

        if frames_processed % 100 == 0:
            print(f"Procesados: {frames_processed} fotogramas")
finally:
    writer.release()

print(f"Listo. Fotogramas procesados: {frames_processed}")
print(f"Video guardado en: {OUTPUT_PATH.resolve()}")

Procesados: 100 fotogramas
Procesados: 200 fotogramas
Procesados: 300 fotogramas
Procesados: 400 fotogramas
Procesados: 500 fotogramas
Procesados: 600 fotogramas
Procesados: 700 fotogramas
Procesados: 800 fotogramas
Procesados: 900 fotogramas
Procesados: 1000 fotogramas
Procesados: 1100 fotogramas
Procesados: 1200 fotogramas
Procesados: 1300 fotogramas
Procesados: 1400 fotogramas
Procesados: 1500 fotogramas
Procesados: 1600 fotogramas
Procesados: 1700 fotogramas
Procesados: 1800 fotogramas
Procesados: 1900 fotogramas
Procesados: 2000 fotogramas
Procesados: 2100 fotogramas
Procesados: 2200 fotogramas
Procesados: 2300 fotogramas
Procesados: 2400 fotogramas
Procesados: 2500 fotogramas
Listo. Fotogramas procesados: 2568
Video guardado en: /Users/alex/Documents/ByteTracker/bytetrack_output.mp4


## 5. Visualizar el resultado
Si el video es largo, el reproductor puede tardar en cargarlo.

In [6]:
display(Video(str(OUTPUT_PATH.resolve()), embed=False, width=800))

## Problemas frecuentes

- **`FileNotFoundError`**: cambia `VIDEO_PATH` por la ruta correcta.
- **No se encuentra `bytetrack.yaml`**: actualiza `ultralytics` y vuelve a ejecutar el notebook.
- **No detecta los objetos esperados**: ByteTrack solo sigue las clases que detecta YOLO. Para objetos específicos puede ser necesario ajustar el modelo.
- **IDs que cambian o se pierden**: depende de la calidad de las detecciones, oclusiones, movimiento de cámara y parámetros del tracker.
- **macOS / Apple Silicon**: empieza con `yolo11n.pt` e `imgsz=640`; aumenta el tamaño si necesitas más detalle y el equipo lo permite.

Esta plantilla utiliza la integración de ByteTrack incluida en Ultralytics. Si buscabas ejecutar el repositorio original de ByteTrack o un archivo `ByteTracker.py` propio, comparte ese código o el enlace del repositorio para adaptar las instrucciones exactamente a esa versión.